# Amazon ML Challenge 2026: Stage 1b — Dedicated Test Dense Embeddings (GPU - 2x T4)
### Dual-Channel Dense Embeddings for Test Source 1, Source 2, and Source 3

**Zero-Duplication Disk-Safe Architecture:**
- Uses `np.lib.format.open_memmap` to write standard `.npy` files directly on disk with zero temporary file duplication.
- Total disk footprint across all 6 files is **17.96 GB** (safely fits within Kaggle's 20.0 GB disk limit).
- Includes instant in-place auto-repair for existing `.tmp` files.
- Resumes seamlessly: skips any file already generated.

In [ ]:
!pip install -q transformers torch pyarrow

import os, sys, gc, time, io
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)} | Count: {torch.cuda.device_count()}")

IS_KAGGLE = os.path.exists('/kaggle')

def find_file(pattern):
    search_root = Path('/kaggle/input') if IS_KAGGLE else Path('.')
    matches = list(search_root.rglob(pattern))
    if not matches:
        return None
    matches.sort(key=lambda p: (p.stat().st_size, p.stat().st_mtime), reverse=True)
    return matches[0]

WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('work')
EMB_DIR = WORK_DIR / 'embeddings'
EMB_DIR.mkdir(parents=True, exist_ok=True)

TEST_S1_NORM = find_file('test_s1_norm.parquet')
TEST_S2_NORM = find_file('test_s2_norm.parquet')
TEST_S3_NORM = find_file('test_s3_norm.parquet')

TEST_S1_TSV = find_file('test_source1.tsv')
TEST_S2_TSV = find_file('test_source2.tsv')
TEST_S3_TSV = find_file('test_source3.tsv')

print(f"Test S1 Norm: {TEST_S1_NORM}")
print(f"Test S2 Norm: {TEST_S2_NORM}")
print(f"Test S3 Norm: {TEST_S3_NORM}")
print(f"Embeddings Output: {EMB_DIR}")

In [ ]:
# ============================================================
# AUTO-REPAIR CELL: REPAIR ANY INTERRUPTED S3 ADDR .TMP FILE
# ============================================================

s3_addr_tmp = EMB_DIR / 'test_s3_addr_emb.npy.tmp'
s3_addr_npy = EMB_DIR / 'test_s3_addr_emb.npy'

# Clean up broken 128-byte placeholder if present
if s3_addr_npy.exists() and s3_addr_npy.stat().st_size < 1000:
    print(f"Removing broken empty placeholder: {s3_addr_npy}")
    os.remove(s3_addr_npy)

# In-place convert existing .tmp file to valid .npy (takes ~2 seconds!)
if s3_addr_tmp.exists() and not s3_addr_npy.exists():
    raw_size = s3_addr_tmp.stat().st_size
    n_rows = raw_size // (384 * 2)
    print(f"Found complete {s3_addr_tmp.name} ({n_rows:,} rows, {raw_size/1e6:.1f} MB)!")
    print("Converting in-place to valid .npy (zero extra disk space)...")
    
    dummy_io = io.BytesIO()
    np.lib.format.write_array_header_1_0(dummy_io, {
        'descr': np.lib.format.dtype_to_descr(np.dtype(np.float16)),
        'fortran_order': False,
        'shape': (n_rows, 384)
    })
    hdr = dummy_io.getvalue()
    hdr_len = len(hdr)
    
    chunk_size = 64 * 1024 * 1024
    with open(s3_addr_tmp, 'r+b') as f:
        f.truncate(raw_size + hdr_len)
        pos = raw_size
        while pos > 0:
            read_size = min(pos, chunk_size)
            pos -= read_size
            f.seek(pos)
            buf = f.read(read_size)
            f.seek(pos + hdr_len)
            f.write(buf)
        f.seek(0)
        f.write(hdr)
        
    os.rename(s3_addr_tmp, s3_addr_npy)
    print(f"\u2705 Repaired {s3_addr_npy.name} successfully!")

In [ ]:
# Model Setup & Direct open_memmap Encoder (Zero-Duplication)
MODEL_NAME = 'intfloat/multilingual-e5-small'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Loading tokenizer and model: {MODEL_NAME} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
base_model.eval()

if DEVICE == 'cuda' and torch.cuda.device_count() > 1:
    print(f"Enabling DataParallel across {torch.cuda.device_count()} GPUs...")
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

@torch.no_grad()
def encode_texts_streaming(texts, out_path, prefix="passage: ", batch_size=512, max_len=64):
    """Encodes texts directly into a standard .npy file via open_memmap with ZERO disk duplication."""
    out_path = Path(out_path)
    if out_path.exists() and out_path.stat().st_size > 1_000_000:
        print(f"  [Already Exists] {out_path.name} ({out_path.stat().st_size / 1e6:.1f} MB) -> SKIPPING")
        return

    n_total = len(texts)
    dim = 384
    print(f"  Encoding {n_total:,} texts -> {out_path.name}...")
    t0 = time.time()
    
    # Directly create standard valid .npy file on disk
    mmap_arr = np.lib.format.open_memmap(out_path, mode='w+', dtype=np.float16, shape=(n_total, dim))
    
    use_amp = (DEVICE == 'cuda')
    for start_idx in range(0, n_total, batch_size):
        end_idx = min(start_idx + batch_size, n_total)
        batch = [f"{prefix}{t}" for t in texts[start_idx:end_idx]]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(DEVICE)
        
        with torch.amp.autocast(device_type=DEVICE, enabled=use_amp):
            out = model(**enc)
            emb = mean_pooling(out, enc['attention_mask'])
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            
        mmap_arr[start_idx:end_idx] = emb.cpu().to(torch.float16).numpy()
        
        if (start_idx // batch_size) % 100 == 0 or end_idx == n_total:
            elapsed = time.time() - t0
            qps = end_idx / max(elapsed, 0.1)
            print(f"    {end_idx:,} / {n_total:,} ({end_idx/n_total*100:.1f}%) | {qps:,.0f} texts/s | Elapsed: {elapsed:.0f}s")
            
    mmap_arr.flush()
    del mmap_arr
    gc.collect()
    print(f"  Saved: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB) in {time.time()-t0:.1f}s")

In [ ]:
# Load Normalized Text Columns for S1, S2, S3
def load_text_columns(norm_path, raw_tsv_path):
    if norm_path and Path(norm_path).exists():
        print(f"Loading from normalized parquet: {norm_path}")
        df = pd.read_parquet(norm_path, columns=['name_core', 'addr_core'])
        names = df['name_core'].fillna('').astype(str).tolist()
        addrs = df['addr_core'].fillna('').astype(str).tolist()
        return names, addrs
    elif raw_tsv_path and Path(raw_tsv_path).exists():
        print(f"Normalizing on-the-fly from raw TSV: {raw_tsv_path}")
        import unicodedata, re
        def clean_fast(s):
            if not s or pd.isna(s):
                return ""
            s = unicodedata.normalize('NFKD', str(s))
            s = re.sub(r'[^a-zA-Z0-9\s]', ' ', s)
            return re.sub(r'\s+', ' ', s).strip().lower()
        df = pd.read_csv(raw_tsv_path, sep='\t')
        names = [clean_fast(x) for x in df['business_name'].values]
        addrs = [clean_fast(x) for x in df['business_address'].values]
        return names, addrs
    else:
        raise FileNotFoundError(f"Neither norm parquet nor raw TSV found!")

# 1. Test S1 (Queries)
print("\n" + "=" * 50)
print(" PROCESSING TEST S1 (QUERIES)")
print("=" * 50)
s1_names, s1_addrs = load_text_columns(TEST_S1_NORM, TEST_S1_TSV)
encode_texts_streaming(s1_names, EMB_DIR / 'test_s1_name_emb.npy', prefix="query: ")
encode_texts_streaming(s1_addrs, EMB_DIR / 'test_s1_addr_emb.npy', prefix="query: ")
del s1_names, s1_addrs; gc.collect()

# 2. Test S2 (Targets)
print("\n" + "=" * 50)
print(" PROCESSING TEST S2 (TARGETS)")
print("=" * 50)
s2_names, s2_addrs = load_text_columns(TEST_S2_NORM, TEST_S2_TSV)
encode_texts_streaming(s2_names, EMB_DIR / 'test_s2_name_emb.npy', prefix="passage: ")
encode_texts_streaming(s2_addrs, EMB_DIR / 'test_s2_addr_emb.npy', prefix="passage: ")
del s2_names, s2_addrs; gc.collect()

# 3. Test S3 (Targets)
print("\n" + "=" * 50)
print(" PROCESSING TEST S3 (TARGETS)")
print("=" * 50)
s3_names, s3_addrs = load_text_columns(TEST_S3_NORM, TEST_S3_TSV)
encode_texts_streaming(s3_names, EMB_DIR / 'test_s3_name_emb.npy', prefix="passage: ")
encode_texts_streaming(s3_addrs, EMB_DIR / 'test_s3_addr_emb.npy', prefix="passage: ")
del s3_names, s3_addrs; gc.collect()

# Verify all 6 embedding files
print("\n" + "=" * 50)
print(" FINAL VERIFICATION OF ALL 6 EMBEDDING FILES")
print("=" * 50)
for fname in [
    'test_s1_name_emb.npy', 'test_s1_addr_emb.npy',
    'test_s2_name_emb.npy', 'test_s2_addr_emb.npy',
    'test_s3_name_emb.npy', 'test_s3_addr_emb.npy'
]:
    p = EMB_DIR / fname
    arr = np.load(p, mmap_mode='r')
    print(f"  {fname:22s} | Shape: {str(arr.shape):16s} | Size: {p.stat().st_size / 1e6:7.1f} MB")

print("\n✅ ALL 6 TEST EMBEDDINGS 100% COMPLETE & VERIFIED!")